In [3]:
import torch
import torch.nn as  nn

In [4]:
a = torch.empty(1)
print(a)

tensor([-2.6677e-23])


In [148]:
# how requires grad work under the hood when forward pass

x = torch.randn((100, 384), requires_grad=True)
y = x + 3
z = y / 3

print(x)
print(x.grad_fn)
print()
print(y)
print(y.grad_fn)
print()
print(z)
print(z._grad_fn)

tensor([[-0.5161, -0.9264,  1.0523,  ...,  0.7762,  0.7987, -0.9763],
        [ 1.2928, -0.5555,  0.9973,  ..., -0.1233, -1.5304, -0.1160],
        [ 1.9707, -0.6957, -1.2375,  ...,  1.9350,  2.5462,  0.5781],
        ...,
        [ 0.6987, -1.5893,  0.0571,  ...,  0.6328, -0.4146, -0.1512],
        [-0.9449,  0.1514,  0.8613,  ...,  0.5792,  0.4609,  0.3588],
        [ 0.5304,  1.4187, -0.1982,  ...,  0.5764, -0.4869,  0.2517]],
       requires_grad=True)
None

tensor([[2.4839, 2.0736, 4.0523,  ..., 3.7762, 3.7987, 2.0237],
        [4.2928, 2.4445, 3.9973,  ..., 2.8767, 1.4696, 2.8840],
        [4.9707, 2.3043, 1.7625,  ..., 4.9350, 5.5462, 3.5781],
        ...,
        [3.6987, 1.4107, 3.0571,  ..., 3.6328, 2.5854, 2.8488],
        [2.0551, 3.1514, 3.8613,  ..., 3.5792, 3.4609, 3.3588],
        [3.5304, 4.4187, 2.8018,  ..., 3.5764, 2.5131, 3.2517]],
       grad_fn=<AddBackward0>)

tensor([[0.8280, 0.6912, 1.3508,  ..., 1.2587, 1.2662, 0.6746],
        [1.4309, 0.8148, 1.3324,  ..., 

In [149]:
# how backpropagation and .grad populate
# gradient value (.grad) will be populated by the Autograd engine, but only at the leaf tensor (the beggining input (the one with requires_grad=True)) if not specified .retain_grad() in the non-leaf tensor
print(x.grad)
loss = z.mean()
loss.backward()
print(x.grad)

None
tensor([[8.6806e-06, 8.6806e-06, 8.6806e-06,  ..., 8.6806e-06, 8.6806e-06,
         8.6806e-06],
        [8.6806e-06, 8.6806e-06, 8.6806e-06,  ..., 8.6806e-06, 8.6806e-06,
         8.6806e-06],
        [8.6806e-06, 8.6806e-06, 8.6806e-06,  ..., 8.6806e-06, 8.6806e-06,
         8.6806e-06],
        ...,
        [8.6806e-06, 8.6806e-06, 8.6806e-06,  ..., 8.6806e-06, 8.6806e-06,
         8.6806e-06],
        [8.6806e-06, 8.6806e-06, 8.6806e-06,  ..., 8.6806e-06, 8.6806e-06,
         8.6806e-06],
        [8.6806e-06, 8.6806e-06, 8.6806e-06,  ..., 8.6806e-06, 8.6806e-06,
         8.6806e-06]])


In [169]:
# to stop the leaf tensor from tracking. for ex: if you want to evaluate the model.
# do either
new_x = x.detach() # this will create the same tensor with require grad to false
# or
x.requires_grad_(False) # this will update the flag inside the leaf tensor
    
    
# the third way is with torch.no_grad()
ex4_a = torch.rand(2, 2, requires_grad=True)
print(ex4_a.requires_grad)

with torch.no_grad():
    ex4_b = ex4_a * 2
    print(ex4_b.requires_grad)

True
False


In [12]:
## linear regression try try
data = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32)
truth = torch.tensor([150, 250, 350, 450, 550], dtype=torch.float32)

# ai brain
weight = torch.zeros(1, requires_grad=True)
bias = torch.zeros(1, requires_grad=True)

print(weight, bias)

def forward(input):
    return input * weight + bias

def loss(truth, prediction):
    return ((truth - prediction) ** 2).mean()

###

epochs = 500
learning_rate = 0.06

for epoch in range(epochs):
    output = forward(data)
    
    l = loss(truth, output)
    
    l.backward()
    
    with torch.no_grad():
        weight -= learning_rate * weight.grad
        bias -= learning_rate * bias.grad
        
    weight.grad.zero_()
    bias.grad.zero_()
    
    if epoch % 20 == 0:
        print(f'epoch {epoch+1} = weight: {weight.item():.3f} | bias: {bias.item():.3f} loss: {l.item():.3f}')
        
print(f'prediction after training: f(5) = {forward(5).item():.3f}')

print(weight, bias)

tensor([0.], requires_grad=True) tensor([0.], requires_grad=True)
epoch 1 = weight: 150.000 | bias: 42.000 loss: 142500.000
epoch 21 = weight: 103.731 | bias: 36.531 loss: 34.400
epoch 41 = weight: 102.476 | bias: 41.061 loss: 15.154
epoch 61 = weight: 101.643 | bias: 44.067 loss: 6.676
epoch 81 = weight: 101.091 | bias: 46.062 loss: 2.941
epoch 101 = weight: 100.724 | bias: 47.386 loss: 1.296
epoch 121 = weight: 100.481 | bias: 48.265 loss: 0.571
epoch 141 = weight: 100.319 | bias: 48.849 loss: 0.251
epoch 161 = weight: 100.212 | bias: 49.236 loss: 0.111
epoch 181 = weight: 100.141 | bias: 49.493 loss: 0.049
epoch 201 = weight: 100.093 | bias: 49.663 loss: 0.021
epoch 221 = weight: 100.062 | bias: 49.777 loss: 0.009
epoch 241 = weight: 100.041 | bias: 49.852 loss: 0.004
epoch 261 = weight: 100.027 | bias: 49.902 loss: 0.002
epoch 281 = weight: 100.018 | bias: 49.935 loss: 0.001
epoch 301 = weight: 100.012 | bias: 49.957 loss: 0.000
epoch 321 = weight: 100.008 | bias: 49.971 loss: 0.00

In [ ]:
## linear regression in pytorch way

data = torch.tensor([ [1], [2], [3], [4], [5] ], dtype=torch.float32)
truth = torch.tensor([ [10], [], [30], [40], [50] ], dtype=torch.float32)

data_sample, data_feature = data.size()
# print(data_sample, data_feature)

class LinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, input_dim ** 2)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(input_dim ** 2, 1)
        
    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x
    

model = LinearRegression(data_feature, data_feature)

epochs = 2000
learning_rate = 0.01

loss = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    prediction = model(data)
    
    l = loss(truth, prediction)
    
    l.backward()
    
    optimizer.step()
    
    optimizer.zero_grad()
    
    if epoch % 20 == 0:
        print(f'epoch {epoch+1} = prediction: {prediction.detach().mean().item():.3f}, loss: {l.item():.3f}')
    
print(f'evaluation: f({10}) = {model(torch.tensor([10], dtype=torch.float32)).item():.3f}')

epoch 1 = prediction: 1.316, loss: 1106.732
epoch 21 = prediction: 10.286, loss: 647.510
epoch 41 = prediction: 17.503, loss: 386.152
epoch 61 = prediction: 22.322, loss: 269.665
epoch 81 = prediction: 25.539, loss: 217.746
epoch 101 = prediction: 27.686, loss: 194.606
epoch 121 = prediction: 29.120, loss: 184.293
epoch 141 = prediction: 30.077, loss: 179.696
epoch 161 = prediction: 30.716, loss: 177.647
epoch 181 = prediction: 31.143, loss: 176.734
epoch 201 = prediction: 31.428, loss: 176.327
epoch 221 = prediction: 31.618, loss: 176.146
epoch 241 = prediction: 31.745, loss: 176.065
epoch 261 = prediction: 31.830, loss: 176.029
epoch 281 = prediction: 31.886, loss: 176.013
epoch 301 = prediction: 31.924, loss: 176.006
epoch 321 = prediction: 31.949, loss: 176.003
epoch 341 = prediction: 31.966, loss: 176.001
epoch 361 = prediction: 31.977, loss: 176.001
epoch 381 = prediction: 31.985, loss: 176.000
epoch 401 = prediction: 31.990, loss: 176.000
epoch 421 = prediction: 31.993, loss: 17